In [10]:
import numpy as np
import pandas as pd

WINDOW_SIZE = 16
STRIDE = 4
CENTER_OFFSET = 8
BOUNDARY_MARGIN = 4

FALL_CLASSES = ["falling", "falled"]

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Duration:", WINDOW_SIZE / 7.8, "seconds")

Window size: 16
Stride: 4
Duration: 2.0512820512820515 seconds


In [11]:
import numpy as np
import pandas as pd

data = np.load("../dataset/fir_frames.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
sessions = data["session"]
file_ids = data["file_id"]
classes = data["classes"]

print("Dataset loaded successfully!")
print("Classes:", classes)

Dataset loaded successfully!
Classes: ['walk' 'stand' 'standup' 'sit' 'sitdown' 'falling' 'falled' 'lying'
 'sleeping' 'active']


In [12]:
# Original class names from dataset
class_names = [str(c) for c in classes]

fall_ids = [
    class_names.index("falling"),
    class_names.index("falled")
]

print("All classes:", class_names)
print("Fall class IDs:", fall_ids)

All classes: ['walk', 'stand', 'standup', 'sit', 'sitdown', 'falling', 'falled', 'lying', 'sleeping', 'active']
Fall class IDs: [5, 6]


In [13]:
window_starts = []
window_labels = []
window_sessions = []
window_file_ids = []
window_original_labels = []

unique_files = np.unique(file_ids)

for fid in unique_files:

    # Frames belonging to this file
    indices = np.where(file_ids == fid)[0]

    # Split into consecutive chunks
    chunks = np.split(
        indices,
        np.where(np.diff(indices) != 1)[0] + 1
    )

    for chunk in chunks:

        if len(chunk) < WINDOW_SIZE:
            continue

        file_labels = y[chunk]

        session_name = sessions[chunk[0]]

        assert np.all(sessions[chunk] == session_name)

        for start in range(
            0,
            len(chunk) - WINDOW_SIZE + 1,
            STRIDE
        ):

            center = start + CENTER_OFFSET

            center_label = int(file_labels[center])

            # Remove unlabeled centers
            if center_label == -1:
                continue

            # Check label stability around center
            left = center - BOUNDARY_MARGIN
            right = center + BOUNDARY_MARGIN + 1

            local_labels = file_labels[left:right]

            # Exclude centers near label boundaries
            if not np.all(local_labels == center_label):
                continue

            # Binary classification
            binary_label = int(center_label in fall_ids)

            global_start = int(chunk[start])

            window_starts.append(global_start)
            window_labels.append(binary_label)
            window_sessions.append(session_name)
            window_file_ids.append(fid)
            window_original_labels.append(center_label)


window_starts = np.array(window_starts)
window_labels = np.array(window_labels)
window_sessions = np.array(window_sessions)
window_file_ids = np.array(window_file_ids)
window_original_labels = np.array(window_original_labels)

print("Total valid windows:", len(window_starts))
print("Fall windows:", np.sum(window_labels == 1))
print("ADL windows:", np.sum(window_labels == 0))

Total valid windows: 48310
Fall windows: 6075
ADL windows: 42235


In [14]:
assert len(window_starts) == len(window_labels)

assert len(window_starts) == len(window_sessions)

assert len(window_starts) == len(window_file_ids)

assert len(window_starts) == len(window_original_labels)

assert np.all(
    window_labels == np.isin(
        window_original_labels,
        fall_ids
    ).astype(np.int8)
)

# Verify that windows do not cross file boundaries
for start, fid in zip(window_starts, window_file_ids):

    stop = start + WINDOW_SIZE

    assert np.all(file_ids[start:stop] == fid)

print("All window consistency checks passed!")

All window consistency checks passed!


In [15]:
class_distribution = (
    pd.Series([
        class_names[int(label)]
        for label in window_original_labels
    ])
    .value_counts()
    .reindex(class_names, fill_value=0)
)

print("Windows per original class:")

display(
    class_distribution.to_frame(
        "Number of windows"
    )
)

Windows per original class:


,Number of windows
walk,13011
stand,208
standup,11018
sit,4463
sitdown,2418
falling,2899
falled,3176
lying,2788
sleeping,7402
active,927


In [16]:
session_table = pd.crosstab(
    window_sessions,
    window_labels
)

session_table = session_table.reindex(
    columns=[0, 1],
    fill_value=0
)

session_table.columns = [
    "ADL (0)",
    "FALL (1)"
]

session_table["Total"] = (
    session_table["ADL (0)"] +
    session_table["FALL (1)"]
)

print("Windows per session:")

display(session_table)

print("\nTotal:")

display(session_table.sum())

Windows per session:


,ADL (0),FALL (1),Total
row_0,,,
20200626_01,4118,605,4723
20200627_01,2748,414,3162
20200628_01,8459,1405,9864
20200628_02,4520,256,4776
20200629_02,17669,3000,20669
20200630_02,4721,395,5116



Total:


ADL (0)     42235
FALL (1)     6075
Total       48310
dtype: int64

In [17]:
fall_count = np.sum(window_labels == 1)

adl_count = np.sum(window_labels == 0)

total = len(window_labels)

print("Total windows:", total)

if total > 0:

    print(
        f"ADL: {adl_count} "
        f"({adl_count / total * 100:.2f}%)"
    )

    print(
        f"FALL: {fall_count} "
        f"({fall_count / total * 100:.2f}%)"
    )

    print(
        "Fall base rate:",
        round(fall_count / total, 4)
    )

else:

    print("No valid windows generated!")

Total windows: 48310
ADL: 42235 (87.42%)
FALL: 6075 (12.58%)
Fall base rate: 0.1258


In [18]:
windows_metadata = pd.DataFrame({

    "start": window_starts,

    "binary_label": window_labels,

    "original_label": window_original_labels,

    "session": window_sessions,

    "file_id": window_file_ids

})

windows_metadata.to_csv(
    "../dataset/windows_metadata.csv",
    index=False
)

print("Windows metadata saved successfully!")

Windows metadata saved successfully!
